##Json parsing and processing

In [2]:
import json
import os 
os.makedirs('data/json_files', exist_ok=True)

In [3]:
# Sample nested JSON data
json_data = {
    "company": "TechCorp",
    "employees": [
        {
            "id": 1,
            "name": "John Doe",
            "role": "Software Engineer",
            "skills": ["Python", "JavaScript", "React"],
            "projects": [
                {"name": "RAG System", "status": "In Progress"},
                {"name": "Data Pipeline", "status": "Completed"}
            ]
        },
        {
            "id": 2,
            "name": "Jane Smith",
            "role": "Data Scientist",
            "skills": ["Python", "SQL", "Machine Learning"],
            "projects": [
                {"name": "Customer Analytics", "status": "Completed"},
                {"name": "Predictive Model", "status": "In Progress"}
            ]
        },
        {
            "id": 3,
            "name": "Bob Johnson",
            "role": "DevOps Engineer",
            "skills": ["Docker", "Kubernetes", "AWS"],
            "projects": [
                {"name": "Infrastructure Setup", "status": "Completed"},
                {"name": "CI/CD Pipeline", "status": "In Progress"}
            ]
        }
    ],
    "departments": ["Engineering", "Data Science", "DevOps"],
    "location": "San Francisco"
}

# Display the JSON data
print("JSON Data Structure:")
print(f"Company: {json_data['company']}")
print(f"Location: {json_data['location']}")
print(f"Total Employees: {len(json_data['employees'])}")
print(f"Departments: {', '.join(json_data['departments'])}")

print("\nEmployee Details:")
for emp in json_data['employees']:
    print(f"\nName: {emp['name']}")
    print(f"Role: {emp['role']}")
    print(f"Skills: {', '.join(emp['skills'])}")
    print("Projects:")
    for project in emp['projects']:
        print(f"  - {project['name']}: {project['status']}")

JSON Data Structure:
Company: TechCorp
Location: San Francisco
Total Employees: 3
Departments: Engineering, Data Science, DevOps

Employee Details:

Name: John Doe
Role: Software Engineer
Skills: Python, JavaScript, React
Projects:
  - RAG System: In Progress
  - Data Pipeline: Completed

Name: Jane Smith
Role: Data Scientist
Skills: Python, SQL, Machine Learning
Projects:
  - Customer Analytics: Completed
  - Predictive Model: In Progress

Name: Bob Johnson
Role: DevOps Engineer
Skills: Docker, Kubernetes, AWS
Projects:
  - Infrastructure Setup: Completed
  - CI/CD Pipeline: In Progress


In [4]:
import json

# Save JSON data to file
with open('data/json_files/company_data.json', 'w') as f:
    json.dump(json_data, f, indent=2)
    
print("✅ JSON data saved to 'data/json_files/company_data.json'")

✅ JSON data saved to 'data/json_files/company_data.json'


In [5]:
import json

# Save JSON Lines format
jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123},
    {"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01", "event": "purchase", "user_id": 123, "amount": 99.99}
]

with open('data/json_files/events.jsonl', 'w') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

print("✅ JSON Lines data saved to 'data/json_files/events.jsonl'")

✅ JSON Lines data saved to 'data/json_files/events.jsonl'


Json Processing Stratergies 

In [ ]:
from langchain_community.document_loaders import JSONLoader
import json


In [1]:
# Method 1: JSONLoader with jq_schema
# This method uses jq queries to extract specific fields from JSON
print("\n1 JSONLoader - Extract specific fields")

from langchain_community.document_loaders import JSONLoader

# Extract employee information
# jq_schema='.employees[0]' extracts the first employee
# You can change the index to get different employees
employee_loader = JSONLoader(
    file_path='data/json_files/company_data.json',  # Path to JSON file
    jq_schema='.employees[0]',  # jq query to extract first employee object
    text_content=False  # Set to False to get full JSON objects as content
)

# Load the documents
employee_docs = employee_loader.load()

# Display results
print(f"Loaded {len(employee_docs)} employee documents")
print(f"\nFirst employee content:")
print(employee_docs[0].page_content)  # Full JSON object as string
print(f"\nMetadata: {employee_docs[0].metadata}")

# Extract all employees (modify jq_schema)
print("\n" + "="*50)
print("Extracting all employees:")

all_employees_loader = JSONLoader(
    file_path='data/json_files/company_data.json',
    jq_schema='.employees[]',  # [] extracts ALL employees
    text_content=False
)

all_employee_docs = all_employees_loader.load()
print(f"Loaded {len(all_employee_docs)} employee documents")

# Show first employee from all extraction
print(f"\nFirst employee (from all extraction):")
print(all_employee_docs[0].page_content)


1 JSONLoader - Extract specific fields


C:\Users\Manish\AppData\Local\Temp\ipykernel_30216\2809701069.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import JSONLoader
c:\Users\Manish\OneDrive\Desktop\MANISH_2026\AI_WORKSETUP\langchain_init_proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 1 employee documents

First employee content:
{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}

Metadata: {'source': 'C:\\Users\\Manish\\OneDrive\\Desktop\\MANISH_2026\\AI_WORKSETUP\\langchain_init_proj\\Data-ingestionParsing\\data\\json_files\\company_data.json', 'seq_num': 1}

Extracting all employees:
Loaded 3 employee documents

First employee (from all extraction):
{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}


In [2]:
# Method 2: Custom JSON processing for complex structures
from typing import List
from langchain_core.documents import Document
import json

print("\n2 Custom JSON Processing")

def process_json_intelligently(filepath: str) -> List[Document]:
    """Process JSON with intelligent flattening and context preservation"""
    
    # Load JSON data from file
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    documents = []
    
    # Strategy 1: Create documents for each employee with full context
    for emp in data.get('employees', []):
        # Build structured content with all employee details
        content = f"""Employee Profile:
Name: {emp['name']}
Role: {emp['role']}
Skills: {', '.join(emp['skills'])}

Projects:
{chr(10).join([f"  - {p['name']}: {p['status']}" for p in emp['projects']])}
"""
        
        # Create Document with enriched metadata
        doc = Document(
            page_content=content,
            metadata={
                'source': filepath,
                'employee_id': emp['id'],
                'name': emp['name'],
                'role': emp['role'],
                'company': data.get('company', 'Unknown'),
                'location': data.get('location', 'Unknown')
            }
        )
        documents.append(doc)
    
    # Strategy 2: Create summary document for company overview
    summary_content = f"""Company Overview:
Company: {data.get('company', 'N/A')}
Location: {data.get('location', 'N/A')}
Total Employees: {len(data.get('employees', []))}
Departments: {', '.join(data.get('departments', []))}
"""
    
    summary_doc = Document(
        page_content=summary_content,
        metadata={
            'source': filepath,
            'type': 'company_summary',
            'total_employees': len(data.get('employees', [])),
            'departments': data.get('departments', [])
        }
    )
    documents.append(summary_doc)
    
    return documents

# Use the custom processor
print("Processing JSON intelligently...")
custom_docs = process_json_intelligently('data/json_files/company_data.json')

print(f"Loaded {len(custom_docs)} documents")
print(f"\nSummary Document:")
print(custom_docs[-1].page_content)  # Last document is summary

print(f"\nEmployee Document:")
print(custom_docs[0].page_content)   # First employee document
print(f"\nMetadata: {custom_docs[0].metadata}")


2 Custom JSON Processing
Processing JSON intelligently...
Loaded 4 documents

Summary Document:
Company Overview:
Company: TechCorp
Location: San Francisco
Total Employees: 3
Departments: Engineering, Data Science, DevOps


Employee Document:
Employee Profile:
Name: John Doe
Role: Software Engineer
Skills: Python, JavaScript, React

Projects:
  - RAG System: In Progress
  - Data Pipeline: Completed


Metadata: {'source': 'data/json_files/company_data.json', 'employee_id': 1, 'name': 'John Doe', 'role': 'Software Engineer', 'company': 'TechCorp', 'location': 'San Francisco'}


In [3]:
process_json_intelligently('data/json_files/company_data.json') 

[Document(metadata={'source': 'data/json_files/company_data.json', 'employee_id': 1, 'name': 'John Doe', 'role': 'Software Engineer', 'company': 'TechCorp', 'location': 'San Francisco'}, page_content='Employee Profile:\nName: John Doe\nRole: Software Engineer\nSkills: Python, JavaScript, React\n\nProjects:\n  - RAG System: In Progress\n  - Data Pipeline: Completed\n'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'employee_id': 2, 'name': 'Jane Smith', 'role': 'Data Scientist', 'company': 'TechCorp', 'location': 'San Francisco'}, page_content='Employee Profile:\nName: Jane Smith\nRole: Data Scientist\nSkills: Python, SQL, Machine Learning\n\nProjects:\n  - Customer Analytics: Completed\n  - Predictive Model: In Progress\n'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'employee_id': 3, 'name': 'Bob Johnson', 'role': 'DevOps Engineer', 'company': 'TechCorp', 'location': 'San Francisco'}, page_content='Employee Profile:\nName: Bob Johnson\nRol